# 23. Application option-contract and PnL engine

## 23.1 Scope and frozen design

This notebook preserves the accepted synthetic contract design. It constructs no N24 model, prediction, or policy.

## 23.2 Inputs and chronology

Load the frozen chronology and raw-IV source. Existing N23 exports provide the invariance baseline.

In [1]:
from pathlib import Path
from datetime import timezone
import hashlib
import warnings
import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display, Markdown
warnings.filterwarnings('default')
pd.set_option('display.max_columns', 80)
BASELINE_MASTER = pd.read_csv(Path.cwd() / 'Data' / 'processed' / '23_contract_admissibility_panel.csv', parse_dates=['model_day'])
BASELINE_COST = pd.read_csv(Path.cwd() / 'Data' / 'processed' / '23_cost_grid_panel.csv', parse_dates=['model_day'])
PROJECT = Path.cwd()
DATA, PROCESSED, HOURLY = (PROJECT / 'Data', PROJECT / 'Data' / 'processed', PROJECT / 'Data' / 'usdjpy_hourly_data')
NY_ZONE, UTC = ('America/New_York', 'UTC')
UPSTREAM_NOTEBOOKS = ['14_empirical_data_audit_and_alignment.ipynb', '15_empirical_variable_construction.ipynb', '16_empirical_gc_tdmi_dependence_structure.ipynb', '17_empirical_parametric_calibration_and_forecasting.ipynb', '18_empirical_eventless_counterfactual_and_weights.ipynb', '19_empirical_nonparametric_causal_structure_and_event_weights.ipynb', '20_empirical_event_weight_api.ipynb', '21_empirical_hidden_event_discovery_and_signal_synthesis.ipynb', '22_application_leakage_safe_hidden_state_reconstruction.ipynb']

def file_hash(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
upstream_hash_before = {name: file_hash(PROJECT / name) for name in UPSTREAM_NOTEBOOKS}
daily = pd.read_csv(PROCESSED / 'usdjpy_daily_model_panel.csv', parse_dates=['model_day'])
n16 = pd.read_csv(PROCESSED / '16_empirical_analysis_panel.csv', parse_dates=['model_day'])
stage_spec = [('C', '2012-08-08', '2017-03-27'), ('VALIDATION', '2017-03-28', '2021-11-12'), ('TEST', '2021-11-15', '2026-07-02')]
chronology = pd.concat([n16.loc[n16.model_day.between(start, end), ['model_day']].assign(stage=stage) for stage, start, end in stage_spec], ignore_index=True)
assert chronology.model_day.is_unique and chronology.groupby('stage').size().to_dict() == {'C': 1209, 'VALIDATION': 1209, 'TEST': 1209}
iv = daily[['model_day', 'atm_vol_quote_points', 'implied_vol_decimal', 'iv_timestamp_status']].rename(columns={'model_day': 'pricing_iv_source_model_day', 'atm_vol_quote_points': 'iv_raw_pct', 'implied_vol_decimal': 'sigma_raw_source_dec'})
assert iv.pricing_iv_source_model_day.is_unique
panel = chronology.merge(iv, left_on='model_day', right_on='pricing_iv_source_model_day', how='left', validate='one_to_one')
panel['sigma_mid_dec'] = pd.to_numeric(panel.iv_raw_pct, errors='coerce') / 100
panel['next_model_day'] = panel.model_day.shift(-1)
panel['state_boundary_ny'] = pd.to_datetime(panel.model_day).dt.tz_localize(NY_ZONE) + pd.Timedelta(hours=17)
panel['state_boundary_utc'] = panel.state_boundary_ny.dt.tz_convert(UTC)
next_boundary = pd.to_datetime(panel.next_model_day).dt.tz_localize(NY_ZONE) + pd.Timedelta(hours=17)
panel['next_observed_model_day_gap_hours'] = (next_boundary - panel.state_boundary_ny).dt.total_seconds() / 3600
panel['nonstandard_next_session_gap'] = panel.next_observed_model_day_gap_hours.gt(26)
assert str(panel.state_boundary_ny.dt.tz) == NY_ZONE and len(panel) == 3627



## 23.3 Mechanical helper functions

Prepare valid observed quotes and deterministic rejection ordering.

In [2]:
def side(name, value):
    frame = pd.read_csv(HOURLY / name)
    frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True, errors='coerce')
    frame[value] = pd.to_numeric(frame.close, errors='coerce')
    return frame[['timestamp', value]]
raw = side('usdjpy_hourly_bid.csv', 'bid').merge(side('usdjpy_hourly_ask.csv', 'ask'), on='timestamp', how='outer', validate='one_to_one').sort_values('timestamp').reset_index(drop=True)
raw['mid'] = (raw.bid + raw.ask) / 2
raw['valid'] = raw.timestamp.notna() & raw.bid.gt(0) & raw.ask.ge(raw.bid) & raw.mid.gt(0) & raw.timestamp.diff().fillna(pd.Timedelta(seconds=1)).gt(pd.Timedelta(0))
quotes = raw.loc[raw.valid, ['timestamp', 'bid', 'ask', 'mid']].reset_index(drop=True)
times = quotes.timestamp.to_numpy(dtype='datetime64[ns]')
assert quotes.timestamp.is_monotonic_increasing and quotes.timestamp.is_unique
REASONS = ['MISSING_IV_T_RAW', 'IV_MODEL_DAY_MISMATCH', 'FRIDAY_ENTRY', 'NO_ENTRY_QUOTE', 'ENTRY_DELAY_TOO_LONG', 'NO_EXPIRY_QUOTE', 'EXPIRY_DELAY_TOO_LONG', 'MATURITY_OUTSIDE_23_25H', 'INSUFFICIENT_HOURLY_QUOTES', 'INTERIOR_GAP_TOO_LARGE']

def after(t):
    i = int(np.searchsorted(times, np.datetime64(t.to_datetime64()), side='left'))
    return quotes.iloc[i] if i < len(quotes) else None

def nearest(t):
    i = int(np.searchsorted(times, np.datetime64(t.to_datetime64()), side='left'))
    z = [quotes.iloc[j] for j in (i - 1, i) if 0 <= j < len(quotes)]
    return min(z, key=lambda x: abs(x.timestamp - t)) if z else None



## 23.4 PHASE A — contract admissibility construction

Construct the mechanical panel without N22 state fields, then audit entry, expiry, Friday, maturity and quote coverage.

In [3]:
def audit(row):
    reasons, out = ([], {})
    if not (pd.notna(row.iv_raw_pct) and np.isfinite(row.sigma_mid_dec) and (row.sigma_mid_dec > 0)):
        reasons.append('MISSING_IV_T_RAW')
    if pd.notna(row.pricing_iv_source_model_day) and row.pricing_iv_source_model_day != row.model_day:
        reasons.append('IV_MODEL_DAY_MISMATCH')
    ent = after(row.state_boundary_utc)
    out.update(entry_timestamp=pd.NaT, entry_delay_minutes=np.nan, entry_bid=np.nan, entry_ask=np.nan, entry_mid=np.nan, intended_expiry=pd.NaT, actual_expiry=pd.NaT, expiry_timing_error_minutes=np.nan, maturity_hours=np.nan, T_years=np.nan, n_valid_spot_observations=0, max_quote_gap_hours=np.nan, invalid_source_rows_encountered=0)
    if ent is None:
        reasons.append('NO_ENTRY_QUOTE')
    else:
        delay = (ent.timestamp - row.state_boundary_utc).total_seconds() / 60
        out.update(entry_timestamp=ent.timestamp, entry_delay_minutes=delay, entry_bid=ent.bid, entry_ask=ent.ask, entry_mid=ent.mid)
        if delay > 60:
            reasons.append('ENTRY_DELAY_TOO_LONG')
    out['is_friday_state_boundary'] = bool(row.state_boundary_ny.weekday() == 4)
    out['actual_entry_weekday'] = ent.timestamp.tz_convert(NY_ZONE).day_name() if ent is not None else pd.NA
    if out['is_friday_state_boundary']:
        reasons.append('FRIDAY_ENTRY')
    if ent is not None:
        intended = ent.timestamp + pd.Timedelta(hours=24)
        exp = nearest(intended)
        out['intended_expiry'] = intended
        if exp is None:
            reasons.append('NO_EXPIRY_QUOTE')
        else:
            err = abs((exp.timestamp - intended).total_seconds() / 60)
            maturity = (exp.timestamp - ent.timestamp).total_seconds() / 3600
            path = quotes.loc[(quotes.timestamp >= ent.timestamp) & (quotes.timestamp <= exp.timestamp)]
            max_gap = path.timestamp.diff().dt.total_seconds().div(3600).max() if len(path) > 1 else np.inf
            invalid = raw.loc[(raw.timestamp >= ent.timestamp) & (raw.timestamp <= exp.timestamp), 'valid'].eq(False).sum()
            out.update(actual_expiry=exp.timestamp, expiry_timing_error_minutes=err, maturity_hours=maturity, T_years=maturity / (365 * 24), n_valid_spot_observations=len(path), max_quote_gap_hours=max_gap, invalid_source_rows_encountered=int(invalid))
            if err > 60:
                reasons.append('EXPIRY_DELAY_TOO_LONG')
            if not 23 <= maturity <= 25:
                reasons.append('MATURITY_OUTSIDE_23_25H')
            if len(path) < 18:
                reasons.append('INSUFFICIENT_HOURLY_QUOTES')
            if max_gap > 2:
                reasons.append('INTERIOR_GAP_TOO_LARGE')
    nonfriday = [x for x in reasons if x != 'FRIDAY_ENTRY']
    out['friday_would_pass_all_nonfriday_rules'] = bool(out['is_friday_state_boundary'] and (not nonfriday))
    out['all_rejection_reasons'] = '|'.join(reasons) if reasons else 'ADMISSIBLE'
    out['primary_rejection_reason'] = next((x for x in REASONS if x in reasons), 'ADMISSIBLE')
    out['contract_admissible'] = not reasons
    return out


## 23.5 PHASE A — rejection funnel and quality diagnostics

Join frozen N22 fields only after mechanical admissibility, then report descriptive hidden-entry survival.

In [4]:
panel = pd.concat([panel, pd.DataFrame([audit(x) for x in panel.itertuples(index=False)], index=panel.index)], axis=1)
for x in ['entry_timestamp', 'intended_expiry', 'actual_expiry']:
    panel[x] = pd.to_datetime(panel[x], utc=True)
panel['calendar_year'] = panel.model_day.dt.year
accepted = panel.contract_admissible
assert (panel.loc[accepted, 'pricing_iv_source_model_day'] == panel.loc[accepted, 'model_day']).all()
mechanical_panel = panel.copy()
FORBIDDEN_HIDDEN_FIELDS = {'strict_hidden', 'state', 'episode_id', 'episode_entry', 'Delta_L', 'common_log_response', 'P_R', 'P_I', 'P_I_prev', 'candidate', 'hidden_state_information_timing'}
hidden_fields_present = sorted(set(mechanical_panel.columns) & FORBIDDEN_HIDDEN_FIELDS)
phaseA_hidden_independence_audit = pd.DataFrame([{
    'forbidden_hidden_fields': '|'.join(sorted(FORBIDDEN_HIDDEN_FIELDS)),
    'mechanical_panel_columns': '|'.join(sorted(mechanical_panel.columns)),
    'forbidden_fields_present': '|'.join(hidden_fields_present),
    'pass': len(hidden_fields_present) == 0,
}])
assert phaseA_hidden_independence_audit['pass'].all()

# N22 is joined only after the mechanical contract result is immutable.
n22 = pd.read_csv(PROCESSED / '22_application_state_panel.csv', parse_dates=['model_day', 'previous_model_day'])
entries = pd.read_csv(PROCESSED / '22_episode_entries.csv', parse_dates=['model_day', 'previous_model_day'])
entries['stage'] = entries['stage'].replace({'validation': 'VALIDATION', 'test': 'TEST'})
assert (PROCESSED / '22_export_manifest.csv').exists() and (PROCESSED / '22_primary_specification.csv').exists()
stage_spec = [('C', '2012-08-08', '2017-03-27'), ('VALIDATION', '2017-03-28', '2021-11-12'), ('TEST', '2021-11-15', '2026-07-02')]
chronology = pd.concat([n16.loc[n16.model_day.between(start, end), ['model_day']].assign(stage=stage) for stage, start, end in stage_spec], ignore_index=True)
assert chronology.model_day.is_unique and chronology.groupby('stage').size().to_dict() == {'C': 1209, 'VALIDATION': 1209, 'TEST': 1209}
assert entries.groupby('stage').size().to_dict() == {'C': 87, 'VALIDATION': 46, 'TEST': 85}
iv = daily[['model_day', 'atm_vol_quote_points', 'implied_vol_decimal', 'iv_timestamp_status']].rename(columns={'model_day': 'pricing_iv_source_model_day', 'atm_vol_quote_points': 'iv_raw_pct', 'implied_vol_decimal': 'sigma_raw_source_dec'})
assert iv.pricing_iv_source_model_day.is_unique
panel = chronology.merge(iv, left_on='model_day', right_on='pricing_iv_source_model_day', how='left', validate='one_to_one')
panel['sigma_mid_dec'] = pd.to_numeric(panel.iv_raw_pct, errors='coerce') / 100
state_cols = [x for x in ['model_day', 'strict_hidden', 'state', 'episode_id', 'episode_entry', 'Delta_L', 'common_log_response', 'P_R', 'P_I_prev', 'any_floor_pair', 'hidden_state_information_timing', 'candidate'] if x in n22.columns]
application_panel = mechanical_panel.merge(n22[state_cols], on='model_day', how='left', validate='one_to_one')
application_panel['state_information_available'] = application_panel.model_day.isin(n22.model_day)
application_panel['strict_hidden'] = application_panel.strict_hidden.fillna(False).astype(bool)
application_panel['episode_entry'] = application_panel.episode_entry.fillna(False).astype(bool)
assert set(application_panel.loc[application_panel.episode_entry, 'model_day']) == set(entries.model_day)
assert application_panel.groupby('stage').episode_entry.sum().astype(int).to_dict() == {'C': 87, 'VALIDATION': 46, 'TEST': 85}
panel = application_panel
funnel = panel.groupby('stage', as_index=False).agg(attempted=('model_day', 'size'), admissible=('contract_admissible', 'sum'))
funnel['rejected'] = funnel.attempted - funnel.admissible
quality = panel.groupby(['stage', 'calendar_year'], as_index=False).agg(attempted=('model_day', 'size'), admissible=('contract_admissible', 'sum'), rejected=('contract_admissible', lambda x: (~x).sum()), friday_rejections=('primary_rejection_reason', lambda x: x.eq('FRIDAY_ENTRY').sum()), friday_would_pass_all_nonfriday_rules=('friday_would_pass_all_nonfriday_rules', 'sum'), median_entry_delay_minutes=('entry_delay_minutes', 'median'), maximum_entry_delay_minutes=('entry_delay_minutes', 'max'), median_maturity_hours=('maturity_hours', lambda x: x[panel.loc[x.index, 'contract_admissible']].median()), median_path_observations=('n_valid_spot_observations', lambda x: x[panel.loc[x.index, 'contract_admissible']].median()), max_observed_quote_gap_hours=('max_quote_gap_hours', lambda x: x[panel.loc[x.index, 'contract_admissible']].max()))
reject_by_year = panel.loc[~accepted].groupby(['stage', 'calendar_year', 'primary_rejection_reason'], as_index=False).size().rename(columns={'size': 'n_contracts'})
hidden_audit = panel.loc[panel.episode_entry].groupby('stage', as_index=False).agg(n22_episode_entries=('model_day', 'size'), mechanically_admissible=('contract_admissible', 'sum'))
hidden_audit['mechanically_rejected'] = hidden_audit.n22_episode_entries - hidden_audit.mechanically_admissible
entry_delay_month = panel.assign(calendar_month=panel.model_day.dt.month).groupby(['stage', 'calendar_month'], as_index=False).agg(n_attempted=('model_day', 'size'), median_entry_delay_minutes=('entry_delay_minutes', 'median'), max_entry_delay_minutes=('entry_delay_minutes', 'max'))


C:\Users\Rajiv Nawal\AppData\Local\Temp\ipykernel_37424\1336116825.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  application_panel['strict_hidden'] = application_panel.strict_hidden.fillna(False).astype(bool)
C:\Users\Rajiv Nawal\AppData\Local\Temp\ipykernel_37424\1336116825.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  application_panel['episode_entry'] = application_panel.episode_entry.fillna(False).astype(bool)


## 23.6 PHASE A — closure export

Export the derived Phase-A closure. A failed check prevents pricing.

In [5]:
phaseA = {'source_rows_C_validation_test_equal_1209': panel.groupby('stage').size().to_dict() == {'C': 1209, 'VALIDATION': 1209, 'TEST': 1209}, 'n22_episode_entries_pre_filter_match': panel.groupby('stage').episode_entry.sum().astype(int).to_dict() == {'C': 87, 'VALIDATION': 46, 'TEST': 85}, 'timezone_aware_ny_boundaries': str(panel.state_boundary_ny.dt.tz) == NY_ZONE, 'iv_source_key_unique': iv.pricing_iv_source_model_day.is_unique, 'iv_merge_cardinality_one_to_one': bool(chronology.model_day.is_unique and iv.pricing_iv_source_model_day.is_unique and len(mechanical_panel) == len(chronology)), 'accepted_iv_matches_state_model_day': (panel.loc[accepted, 'pricing_iv_source_model_day'] == panel.loc[accepted, 'model_day']).all(), 'accepted_has_raw_iv': panel.loc[accepted, 'iv_raw_pct'].notna().all(), 'friday_derived_from_state_boundary': (~panel.loc[accepted, 'is_friday_state_boundary']).all(), 'accepted_entry_delay_at_most_60_minutes': panel.loc[accepted, 'entry_delay_minutes'].between(0, 60).all(), 'accepted_expiry_error_at_most_60_minutes': panel.loc[accepted, 'expiry_timing_error_minutes'].between(0, 60).all(), 'accepted_maturity_in_23_to_25_hours': panel.loc[accepted, 'maturity_hours'].between(23, 25).all(), 'accepted_has_at_least_18_observations': panel.loc[accepted, 'n_valid_spot_observations'].ge(18).all(), 'accepted_max_gap_at_most_2_hours': panel.loc[accepted, 'max_quote_gap_hours'].le(2).all(), 'attempted_equals_admissible_plus_rejected': (funnel.attempted == funnel.admissible + funnel.rejected).all(), 'one_row_per_attempted_contract': len(panel) == 3627 and panel.model_day.is_unique, 'phaseA_construction_excludes_hidden_fields': phaseA_hidden_independence_audit['pass'].all()}
phaseA_closure = pd.DataFrame({'check': phaseA.keys(), 'pass': phaseA.values()})
phaseA_closure['phaseA_ready'] = bool(phaseA_closure['pass'].all())
PHASE_A_READY = bool(phaseA_closure['phaseA_ready'].iloc[0])
phaseA_closure.to_csv(PROCESSED / '23_phaseA_closure.csv', index=False)
display(Markdown('## Phase A — mechanical contract admissibility'))
display(funnel)
display(Markdown('### N22 hidden episode-entry survival'))
display(hidden_audit)
display(Markdown('### Primary rejections by stage/year'))
display(reject_by_year)
display(Markdown('### Friday and entry-delay audits'))
display(quality[['stage', 'calendar_year', 'friday_rejections', 'friday_would_pass_all_nonfriday_rules']])
display(entry_delay_month)
display(phaseA_closure)
assert PHASE_A_READY, 'Phase A closure failed; do not begin Phase B.'


## Phase A — mechanical contract admissibility

,stage,attempted,admissible,rejected
0,C,1209,961,248
1,TEST,1209,962,247
2,VALIDATION,1209,961,248


### N22 hidden episode-entry survival

,stage,n22_episode_entries,mechanically_admissible,mechanically_rejected
0,C,87,69,18
1,TEST,85,74,11
2,VALIDATION,46,34,12


### Primary rejections by stage/year

,stage,calendar_year,primary_rejection_reason,n_contracts
0,C,2012,FRIDAY_ENTRY,21
1,C,2013,FRIDAY_ENTRY,52
2,C,2013,INTERIOR_GAP_TOO_LARGE,2
3,C,2014,FRIDAY_ENTRY,52
4,C,2014,INSUFFICIENT_HOURLY_QUOTES,2
5,C,2015,ENTRY_DELAY_TOO_LONG,1
6,C,2015,EXPIRY_DELAY_TOO_LONG,1
7,C,2015,FRIDAY_ENTRY,52
8,C,2016,FRIDAY_ENTRY,53
9,C,2017,FRIDAY_ENTRY,12


### Friday and entry-delay audits

,stage,calendar_year,friday_rejections,friday_would_pass_all_nonfriday_rules
0,C,2012,21,20
1,C,2013,52,52
2,C,2014,52,18
3,C,2015,52,0
4,C,2016,53,0
5,C,2017,12,0
6,TEST,2021,7,0
7,TEST,2022,52,0
8,TEST,2023,52,0
9,TEST,2024,52,0


,stage,calendar_month,n_attempted,median_entry_delay_minutes,max_entry_delay_minutes
0,C,1,111,0.0,2880.0
1,C,2,101,0.0,2880.0
2,C,3,106,0.0,2880.0
3,C,4,87,0.0,2880.0
4,C,5,88,0.0,2880.0
5,C,6,85,0.0,2880.0
6,C,7,90,0.0,2880.0
7,C,8,105,0.0,2880.0
8,C,9,107,0.0,2880.0
9,C,10,112,0.0,2940.0


,check,pass,phaseA_ready
0,source_rows_C_validation_test_equal_1209,True,True
1,n22_episode_entries_pre_filter_match,True,True
2,timezone_aware_ny_boundaries,True,True
3,iv_source_key_unique,True,True
4,iv_merge_cardinality_one_to_one,True,True
5,accepted_iv_matches_state_model_day,True,True
6,accepted_has_raw_iv,True,True
7,friday_derived_from_state_boundary,True,True
8,accepted_entry_delay_at_most_60_minutes,True,True
9,accepted_expiry_error_at_most_60_minutes,True,True


## 23.7 PHASE B — gate reload

Reload the exported closure from disk; this is a real restart-safe gate.

In [6]:
phaseA_disk = pd.read_csv(PROCESSED / '23_phaseA_closure.csv')
assert phaseA_disk['pass'].all() and phaseA_disk['phaseA_ready'].eq(True).all(), 'Phase A closure failed on disk.'



## 23.8 Garman-Kohlhagen helpers and pricing audits

At zero rates and spot ATM, forward equals spot by model assumption. Call-put equality is audited numerically.

In [7]:
def gk(S, K, sigma, T, rd=0.0, rf=0.0):
    if not (S > 0 and K > 0 and (sigma > 0) and (T > 0)):
        raise ValueError('Invalid Garman-Kohlhagen input')
    d1 = (np.log(S / K) + (rd - rf + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    df = np.exp(-rf * T)
    dd = np.exp(-rd * T)
    return (S * df * norm.cdf(d1) - K * dd * norm.cdf(d2), K * dd * norm.cdf(-d2) - S * df * norm.cdf(-d1), df * (norm.cdf(d1) - norm.cdf(-d1)))

def trade_cost(dq, b, a):
    return abs(dq) * (a - b) / 2
pnls, ledger = ({}, [])
for r in panel.loc[accepted].itertuples():
    i = r.Index
    path = quotes.loc[(quotes.timestamp >= r.entry_timestamp) & (quotes.timestamp <= r.actual_expiry)].reset_index(drop=True)
    call, put, _ = gk(r.entry_mid, r.entry_mid, r.sigma_mid_dec, r.T_years)
    V0 = call + put
    delta = np.array([gk(path.mid[j], r.entry_mid, r.sigma_mid_dec, (r.actual_expiry - path.timestamp[j]).total_seconds() / (365 * 24 * 3600))[2] for j in range(len(path) - 1)])
    hedge = float(np.sum(-delta * np.diff(path.mid.to_numpy())))
    payoff = abs(path.mid.iloc[-1] - r.entry_mid)
    gross = payoff - V0 + hedge
    ql, qs = (-delta, delta)
    prev_l = prev_s = cost_l = cost_s = 0.0
    init_l = init_s = np.nan
    for j in range(len(delta)):
        dl, ds = (ql[j] - prev_l, qs[j] - prev_s)
        cl, cs = (trade_cost(dl, path.bid[j], path.ask[j]), trade_cost(ds, path.bid[j], path.ask[j]))
        cost_l += cl
        cost_s += cs
        if j == 0:
            init_l, init_s = (cl, cs)
        ledger.append({'model_day': r.model_day, 'ledger_time': path.timestamp[j], 'ledger_event': 'INITIAL_HEDGE' if j == 0 else 'REBALANCE', 'long_trade': dl, 'short_trade': ds, 'long_spot_cost': cl, 'short_spot_cost': cs})
        prev_l, prev_s = (ql[j], qs[j])
    close_l, close_s = (trade_cost(-prev_l, path.bid.iloc[-1], path.ask.iloc[-1]), trade_cost(-prev_s, path.bid.iloc[-1], path.ask.iloc[-1]))
    cost_l += close_l
    cost_s += close_s
    ledger.append({'model_day': r.model_day, 'ledger_time': path.timestamp.iloc[-1], 'ledger_event': 'TERMINAL_UNWIND', 'long_trade': -prev_l, 'short_trade': -prev_s, 'long_spot_cost': close_l, 'short_spot_cost': close_s})
    pnls[i] = {'K': r.entry_mid, 'V0_mid': V0, 'payoff_T': payoff, 'hedge_pnl_mid': hedge, 'Pi_gross': gross, 'Y_gross': gross / V0, 'c_initial_hedge_L': init_l / V0, 'c_initial_hedge_S': init_s / V0, 'c_spot_total_L': cost_l / V0, 'c_spot_total_S': cost_s / V0, 'gross_short_Pi': -gross}
panel = panel.join(pd.DataFrame.from_dict(pnls, orient='index'))
hedge_ledger = pd.DataFrame(ledger)
assert panel.loc[accepted, 'V0_mid'].gt(0).all()


## 23.9 Gross delta-hedged PnL

Use observed quote paths, fixed entry IV deltas, and actual elapsed maturity.

In [8]:
scenarios = [('LOW', 0.1), ('BASE', 0.2), ('HIGH', 0.4)]
costs = []
for r in panel.loc[accepted].itertuples(index=False):
    for label, sp in scenarios:
        sb = r.sigma_mid_dec - sp / 200
        sa = r.sigma_mid_dec + sp / 200
        assert sb > 0
        vb = sum(gk(r.entry_mid, r.K, sb, r.T_years)[:2])
        vm = sum(gk(r.entry_mid, r.K, r.sigma_mid_dec, r.T_years)[:2])
        va = sum(gk(r.entry_mid, r.K, sa, r.T_years)[:2])
        col = (va - vm) / vm
        cos = (vm - vb) / vm
        costs.append({'model_day': r.model_day, 'stage': r.stage, 'state': r.state, 'strict_hidden': r.strict_hidden, 'episode_id': r.episode_id, 'episode_entry': r.episode_entry, 's_sigma_label': label, 's_sigma_vol_points': sp, 'sigma_bid_dec': sb, 'sigma_mid_dec': r.sigma_mid_dec, 'sigma_ask_dec': sa, 'V0_bid': vb, 'V0_mid': vm, 'V0_ask': va, 'c_opt_L': col, 'c_opt_S': cos, 'c_initial_hedge_L': r.c_initial_hedge_L, 'c_initial_hedge_S': r.c_initial_hedge_S, 'c_entry_L': col + r.c_initial_hedge_L, 'c_entry_S': cos + r.c_initial_hedge_S, 'c_spot_total_L': r.c_spot_total_L, 'c_spot_total_S': r.c_spot_total_S, 'Y_gross': r.Y_gross, 'Y_net_L': r.Y_gross - col - r.c_spot_total_L, 'Y_net_S': -r.Y_gross - cos - r.c_spot_total_S})
cost_grid = pd.DataFrame(costs)
assert len(cost_grid) == 3 * accepted.sum() and cost_grid.sigma_bid_dec.gt(0).all()
assert np.allclose(cost_grid.c_entry_L, cost_grid.c_opt_L + cost_grid.c_initial_hedge_L) and np.allclose(cost_grid.c_entry_S, cost_grid.c_opt_S + cost_grid.c_initial_hedge_S)


## 23.10 Transaction-cost ledger

Separate entry-known option/initial-hedge costs from realised future hedge costs.

## 23.11 Development economic diagnostics

Only C and VALIDATION economic diagnostics are displayed; TEST remains export-only.

In [9]:
dev = panel.loc[accepted & panel.stage.isin(['C', 'VALIDATION'])]

def target(x, s):
    v = x.Y_gross
    return {'stage': s, 'n': len(v), 'mean': v.mean(), 'std': v.std(ddof=1), 'median': v.median(), 'min': v.min(), 'max': v.max(), 'q01': v.quantile(0.01), 'q05': v.quantile(0.05), 'q25': v.quantile(0.25), 'q75': v.quantile(0.75), 'q95': v.quantile(0.95), 'q99': v.quantile(0.99), 'fraction_Y_gross_lt_minus_1': (v < -1).mean()}
dev_target = pd.DataFrame([target(dev.loc[dev.stage.eq(s)], s) for s in ['C', 'VALIDATION']])

def costs_summary(x, pop):
    rows = []
    for (s, l), g in x.groupby(['stage', 's_sigma_label']):
        ratio_l = g.c_spot_total_L / g.c_opt_L
        ratio_s = g.c_spot_total_S / g.c_opt_S
        rows.append({'population': pop, 'stage': s, 's_sigma_label': l, 's_sigma_vol_points': g.s_sigma_vol_points.iloc[0], 'n': len(g), 'mean_c_opt_L': g.c_opt_L.mean(), 'median_c_opt_L': g.c_opt_L.median(), 'mean_c_opt_S': g.c_opt_S.mean(), 'median_c_opt_S': g.c_opt_S.median(), 'mean_c_initial_hedge_L': g.c_initial_hedge_L.mean(), 'median_c_initial_hedge_L': g.c_initial_hedge_L.median(), 'mean_c_initial_hedge_S': g.c_initial_hedge_S.mean(), 'median_c_initial_hedge_S': g.c_initial_hedge_S.median(), 'mean_c_spot_total_L': g.c_spot_total_L.mean(), 'median_c_spot_total_L': g.c_spot_total_L.median(), 'mean_c_spot_total_S': g.c_spot_total_S.mean(), 'median_c_spot_total_S': g.c_spot_total_S.median(), 'median_spot_over_option_L': ratio_l.median(), 'median_spot_over_option_S': ratio_s.median(), 'ex_post_friction_materiality_long': (g.Y_gross > g.c_entry_L).mean(), 'ex_post_friction_materiality_short': (g.Y_gross < -g.c_entry_S).mean(), 'diagnostic_label': 'EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY'})
    return rows
dev_cost = pd.DataFrame(costs_summary(cost_grid.loc[cost_grid.stage.isin(['C', 'VALIDATION'])], 'ALL_ADMISSIBLE') + costs_summary(cost_grid.loc[cost_grid.stage.isin(['C', 'VALIDATION']) & cost_grid.episode_entry], 'HIDDEN_EPISODE_ENTRY_ADMISSIBLE'))
display(Markdown('## Phase B — DEVELOPMENT and VALIDATION economics only'))
display(dev_target)
display(dev_cost)
print('These ex-post friction materiality diagnostics are not trade counts. No TEST economic result is displayed.')
priced = panel.loc[accepted]
entry_hurdle_dependency_audit = pd.DataFrame([
    {'side': 'LONG', 'max_abs_residual': float(np.abs(cost_grid.c_entry_L - cost_grid.c_opt_L - cost_grid.c_initial_hedge_L).max()), 'pass': bool(np.allclose(cost_grid.c_entry_L, cost_grid.c_opt_L + cost_grid.c_initial_hedge_L, rtol=0, atol=1e-12))},
    {'side': 'SHORT', 'max_abs_residual': float(np.abs(cost_grid.c_entry_S - cost_grid.c_opt_S - cost_grid.c_initial_hedge_S).max()), 'pass': bool(np.allclose(cost_grid.c_entry_S, cost_grid.c_opt_S + cost_grid.c_initial_hedge_S, rtol=0, atol=1e-12))},
])
cost_grid['entry_hurdle_uses_future_information'] = not bool(entry_hurdle_dependency_audit['pass'].all())
test_discipline_audit = pd.DataFrame([
    {'diagnostic_name': 'development_target_summary', 'stage_values_used': '|'.join(sorted(dev_target.stage.unique()))},
    {'diagnostic_name': 'development_cost_summary', 'stage_values_used': '|'.join(sorted(dev_cost.stage.unique()))},
])
test_discipline_audit['contains_TEST'] = test_discipline_audit.stage_values_used.str.contains('TEST')
test_discipline_audit['pass'] = ~test_discipline_audit.contains_TEST
phaseB = {'phase_a_ready_before_pricing': PHASE_A_READY, 'positive_entry_mid_premium': priced.V0_mid.gt(0).all(), 'positive_scenario_bid_volatility': cost_grid.sigma_bid_dec.gt(0).all(), 'nonnegative_terminal_payoff': priced.payoff_T.ge(0).all(), 'gross_pnl_identity': np.allclose(priced.Pi_gross, priced.payoff_T - priced.V0_mid + priced.hedge_pnl_mid), 'normalisation_identity': np.allclose(priced.Y_gross, priced.Pi_gross / priced.V0_mid), 'long_short_gross_symmetry': np.allclose(priced.gross_short_Pi, -priced.Pi_gross), 'option_spread_charged_once_at_entry': np.allclose(cost_grid.Y_net_L, cost_grid.Y_gross - cost_grid.c_opt_L - cost_grid.c_spot_total_L), 'spot_cost_ledger_has_initial_rebalance_and_terminal': set(hedge_ledger.ledger_event) == {'INITIAL_HEDGE', 'REBALANCE', 'TERMINAL_UNWIND'}, 'entry_hurdle_dependency_audit_pass': bool(entry_hurdle_dependency_audit['pass'].all()),
    'entry_hurdle_excludes_future_hedge_cost': np.allclose(cost_grid.c_entry_L, cost_grid.c_opt_L + cost_grid.c_initial_hedge_L) and (~cost_grid.entry_hurdle_uses_future_information).all(), 'test_economic_results_not_used_for_selection': bool(test_discipline_audit['pass'].all())}
phaseB_closure = pd.DataFrame({'check': phaseB.keys(), 'pass': phaseB.values()})
assert phaseB_closure['pass'].all()
spec = pd.DataFrame([{'state_source': '22_application_leakage_safe_hidden_state_reconstruction.ipynb', 'contract_construction_scope': 'FULL_C_VALIDATION_TEST_CHRONOLOGY', 'hidden_status_used_for_contract_construction': False, 'state_information_boundary': '17:00_America/New_York', 'pricing_iv': 'same_model_day_raw_ATM_IV_t', 'pricing_iv_exact_snapshot_time_known': False, 'missing_iv_rule': 'CONTRACT_INADMISSIBLE', 'entry_rule': 'first_valid_spot_quote_at_or_after_state_boundary', 'entry_max_delay_minutes': 60, 'strike_rule': 'SPOT_ATM_K_EQUALS_ENTRY_MID', 'rd': 0, 'rf': 0, 'friday_entry_rule': 'REJECT', 'intended_holding_hours': 24, 'admissible_maturity_min_hours': 23, 'admissible_maturity_max_hours': 25, 'expiry_quote_max_error_minutes': 60, 'hedge_rule': 'QUOTE_BASED_OBSERVED_HOURLY_NO_INTERPOLATION', 'max_quote_gap_hours': 2, 'minimum_spot_observations': 18, 'hedge_volatility_rule': 'FIXED_ENTRY_IV_RAW', 'normaliser': 'ENTRY_STRADDLE_MID_PREMIUM', 'predictive_target': 'GROSS_DELTA_HEDGED_STRADDLE_PNL_OVER_ENTRY_PREMIUM', 'transaction_costs_inside_predictive_target': False, 'option_spread_charged_at': 'ENTRY_ONLY', 'option_vol_spread_low': 0.1, 'option_vol_spread_base': 0.2, 'option_vol_spread_high': 0.4, 'option_vol_spread_units': 'FULL_BID_ASK_VOLATILITY_PERCENTAGE_POINTS', 'decision_hurdle': 'ENTRY_KNOWN_COSTS_ONLY', 'future_realised_hedge_cost_used_in_entry_decision': False, 'n26_predeclared_sensitivities': '20_observations;0.8x_1.2x_IV_delta;nonzero_rates;low_base_high_vol_grid;expected_future_cost_allowance;IV_snapshot_timing'}])
cols = ['model_day', 'stage', 'state_boundary_ny', 'state_boundary_utc', 'iv_raw_pct', 'sigma_mid_dec', 'pricing_iv_source_model_day', 'iv_timestamp_status', 'entry_timestamp', 'entry_delay_minutes', 'entry_bid', 'entry_ask', 'entry_mid', 'is_friday_state_boundary', 'actual_entry_weekday', 'intended_expiry', 'actual_expiry', 'expiry_timing_error_minutes', 'maturity_hours', 'T_years', 'next_observed_model_day_gap_hours', 'nonstandard_next_session_gap', 'n_valid_spot_observations', 'max_quote_gap_hours', 'invalid_source_rows_encountered', 'contract_admissible', 'primary_rejection_reason', 'all_rejection_reasons', 'friday_would_pass_all_nonfriday_rules', 'state_information_available', 'strict_hidden', 'state', 'episode_id', 'episode_entry', 'Delta_L', 'common_log_response', 'P_R', 'P_I_prev', 'any_floor_pair', 'K', 'V0_mid', 'payoff_T', 'hedge_pnl_mid', 'Pi_gross', 'Y_gross', 'c_initial_hedge_L', 'c_initial_hedge_S', 'c_spot_total_L', 'c_spot_total_S']
master = panel[[x for x in cols if x in panel.columns]]
assert upstream_hash_before == {name: file_hash(PROJECT / name) for name in UPSTREAM_NOTEBOOKS}
iv_provenance = pd.DataFrame([{'source_iv_key_unique': iv.pricing_iv_source_model_day.is_unique, 'application_key_unique': chronology.model_day.is_unique, 'merge_cardinality': 'one_to_one', 'application_days_missing_iv': int(panel.iv_raw_pct.isna().sum()), 'source_model_day_equals_application_day_for_priced_rows': bool((panel.loc[accepted, 'pricing_iv_source_model_day'] == panel.loc[accepted, 'model_day']).all())}])
put_call_residual = []
for record in panel.loc[accepted].itertuples(index=False):
    call_mid, put_mid, _ = gk(record.entry_mid, record.K, record.sigma_mid_dec, record.T_years)
    put_call_residual.append(call_mid - put_mid)
put_call_residual = np.asarray(put_call_residual)
gross_residual = panel.loc[accepted, 'Pi_gross'] - (panel.loc[accepted, 'payoff_T'] - panel.loc[accepted, 'V0_mid'] + panel.loc[accepted, 'hedge_pnl_mid'])
norm_residual = panel.loc[accepted, 'Y_gross'] - panel.loc[accepted, 'Pi_gross'] / panel.loc[accepted, 'V0_mid']
pricing_audit = pd.DataFrame([{'maximum_absolute_put_call_residual': float(np.abs(put_call_residual).max()), 'median_absolute_put_call_residual': float(np.median(np.abs(put_call_residual))), 'put_call_rows_failing_atol_1e_12': int((np.abs(put_call_residual) > 1e-12).sum()), 'maximum_absolute_gross_pnl_residual': float(np.abs(gross_residual).max()), 'maximum_absolute_normalisation_residual': float(np.abs(norm_residual).max())}])
entry_hurdle_dependency_audit = pd.DataFrame([{'side': 'LONG', 'max_abs_residual': float(np.abs(cost_grid.c_entry_L - cost_grid.c_opt_L - cost_grid.c_initial_hedge_L).max()), 'pass': bool(np.allclose(cost_grid.c_entry_L, cost_grid.c_opt_L + cost_grid.c_initial_hedge_L, rtol=0, atol=1e-12))}, {'side': 'SHORT', 'max_abs_residual': float(np.abs(cost_grid.c_entry_S - cost_grid.c_opt_S - cost_grid.c_initial_hedge_S).max()), 'pass': bool(np.allclose(cost_grid.c_entry_S, cost_grid.c_opt_S + cost_grid.c_initial_hedge_S, rtol=0, atol=1e-12))}])
test_discipline_audit = pd.DataFrame([{'diagnostic_name': 'development_target_summary', 'stage_values_used': '|'.join(sorted(dev_target.stage.unique()))}, {'diagnostic_name': 'development_cost_summary', 'stage_values_used': '|'.join(sorted(dev_cost.stage.unique()))}])
test_discipline_audit['contains_TEST'] = test_discipline_audit.stage_values_used.str.contains('TEST')
test_discipline_audit['pass'] = ~test_discipline_audit.contains_TEST
admitted_path_coverage_audit = panel.loc[accepted].groupby('stage').n_valid_spot_observations.agg(['count', 'min', 'median', 'max', lambda x: int(x.eq(18).sum()), lambda x: int(x.eq(19).sum()), lambda x: int(x.lt(20).sum())]).reset_index()
admitted_path_coverage_audit.columns = ['stage', 'n', 'min', 'median', 'max', 'n_eq_18', 'n_eq_19', 'would_fail_hypothetical_20_observation_floor']
friday_would_pass_audit = quality[['stage', 'calendar_year', 'friday_rejections', 'friday_would_pass_all_nonfriday_rules']].rename(columns={'calendar_year': 'year'})
nonfriday_no_entry_audit = panel.assign(no_entry=panel.all_rejection_reasons.str.contains('NO_ENTRY_QUOTE', na=False)).groupby('stage').apply(lambda x: int((~x.is_friday_state_boundary & x.no_entry).sum())).reset_index(name='nonfriday_no_entry_quote_count')
# Display the substantive audits before writing the final manifest.
iv_provenance = pd.DataFrame([{'application_keys_unique': chronology.model_day.is_unique, 'iv_source_keys_unique': iv.pricing_iv_source_model_day.is_unique, 'row_count_preserved': len(mechanical_panel) == len(chronology), 'matched_rows': int(panel.pricing_iv_source_model_day.notna().sum()), 'unmatched_application_rows': int(panel.pricing_iv_source_model_day.isna().sum()), 'missing_iv_rows': int(panel.iv_raw_pct.isna().sum()), 'model_day_mismatch_count': int((panel.loc[panel.pricing_iv_source_model_day.notna(), 'pricing_iv_source_model_day'] != panel.loc[panel.pricing_iv_source_model_day.notna(), 'model_day']).sum()), 'merge_cardinality_pass': bool(chronology.model_day.is_unique and iv.pricing_iv_source_model_day.is_unique and len(mechanical_panel) == len(chronology))}])
master_now = master.copy()
base_master = BASELINE_MASTER.loc[BASELINE_MASTER.contract_admissible].copy()
now_adm = master_now.loc[master_now.contract_admissible].copy()
contract_row_invariant = base_master.set_index('model_day').contract_admissible.equals(now_adm.set_index('model_day').contract_admissible)
admissible_sets_invariant = all(set(base_master.loc[base_master.stage.eq(stage), 'model_day']) == set(now_adm.loc[now_adm.stage.eq(stage), 'model_day']) for stage in ['C','VALIDATION','TEST'])
economic_cols = ['V0_mid','payoff_T','hedge_pnl_mid','Pi_gross','Y_gross']
cmp = now_adm[['model_day'] + economic_cols].merge(base_master[['model_day'] + economic_cols], on='model_day', suffixes=('_now','_base'), validate='one_to_one')
economic_invariant = all(np.allclose(cmp[f'{x}_now'], cmp[f'{x}_base'], rtol=0, atol=1e-12, equal_nan=True) for x in economic_cols)
entry_hurdle_uses_future_information = not bool(entry_hurdle_dependency_audit['pass'].all())
cost_grid['entry_hurdle_uses_future_information'] = entry_hurdle_uses_future_information
for name, frame, explanation in [('pricing_audit', pricing_audit, 'Numerical consistency of the synthetic option-pricing implementation.'), ('admitted_path_coverage_audit', admitted_path_coverage_audit, 'Accepted hedge-path density; the 20-observation sensitivity remains descriptive.'), ('nonfriday_no_entry_audit', nonfriday_no_entry_audit, 'Remaining non-Friday entry failures are a session/data-quality diagnostic.'), ('iv_provenance', iv_provenance, 'Same-model-day raw-IV source provenance and derived merge checks.'), ('entry_hurdle_dependency_audit', entry_hurdle_dependency_audit, 'Only entry-known option and initial-hedge costs enter the hurdle.'), ('test_discipline_audit', test_discipline_audit, 'Development economic diagnostics use C and VALIDATION only.')]:
    display(Markdown(explanation)); display(frame)
print('N23 FINAL FREEZE CLOSURE')
print('Admissible counts:', funnel.set_index('stage').admissible.to_dict())
print('Hidden-entry admissible:', hidden_audit.set_index('stage').mechanically_admissible.to_dict())
print('Contract-row invariant:', contract_row_invariant, 'date-set invariant:', admissible_sets_invariant, 'economic invariant:', economic_invariant)
print('Entry hurdle uses future information:', entry_hurdle_uses_future_information)
exports = {'23_contract_admissibility_panel.csv': master, '23_contract_rejection_funnel.csv': funnel, '23_contract_quality_by_stage_year.csv': quality, '23_hidden_entry_contract_audit.csv': hidden_audit, '23_option_contract_and_gross_pnl_panel.csv': master, '23_cost_grid_panel.csv': cost_grid, '23_development_target_summary.csv': dev_target, '23_development_cost_summary.csv': dev_cost, '23_primary_specification.csv': spec, '23_phaseA_closure.csv': phaseA_closure, '23_phaseB_closure.csv': phaseB_closure, '23_contract_rejections_by_stage_year_reason.csv': reject_by_year.rename(columns={'calendar_year': 'year', 'n_contracts': 'count'}), '23_friday_would_pass_audit.csv': friday_would_pass_audit, '23_nonfriday_no_entry_audit.csv': nonfriday_no_entry_audit, '23_iv_model_day_provenance_audit.csv': iv_provenance, '23_entry_hurdle_dependency_audit.csv': entry_hurdle_dependency_audit, '23_test_discipline_audit.csv': test_discipline_audit, '23_admitted_path_coverage_audit.csv': admitted_path_coverage_audit, '23_pricing_identity_audit.csv': pricing_audit, '23_scientific_invariance_audit.csv': pd.DataFrame([{'contract_admissible_row_invariant': contract_row_invariant, 'admissible_date_sets_invariant': admissible_sets_invariant, 'economic_outputs_invariant_atol_1e_12': economic_invariant}]), '23_phaseA_hidden_independence_audit.csv': phaseA_hidden_independence_audit}
for name, frame in exports.items():
    frame.to_csv(PROCESSED / name, index=False)
manifest = pd.DataFrame([{'file': n, 'rows': len(f), 'exists': (PROCESSED / n).exists(), 'nonempty_file': (PROCESSED / n).stat().st_size > 0} for n, f in exports.items()])
manifest = pd.concat([manifest, pd.DataFrame([{'file': '23_export_manifest.csv', 'rows': len(manifest) + 1, 'exists': True, 'nonempty_file': True}])], ignore_index=True)
manifest.to_csv(PROCESSED / '23_export_manifest.csv', index=False)
display(Markdown('## Final N23 closure report'))
display(funnel)
display(hidden_audit)
display(phaseA_closure)
display(phaseB_closure)
print('PHASE A: PASS | PHASE B: PASS | upstream notebooks modified: NO | TEST economic performance summary displayed: NO | N23 ready for Notebook 24: YES')


## Phase B — DEVELOPMENT and VALIDATION economics only

,stage,n,mean,std,median,min,max,q01,q05,q25,q75,q95,q99,fraction_Y_gross_lt_minus_1
0,C,961,-0.070680,0.346294,-0.087093,-0.920275,2.294433,-0.763089,-0.563700,-0.282984,0.082376,0.512499,1.087925,0.0
1,VALIDATION,961,0.052037,0.474077,-0.013113,-0.871396,6.538297,-0.675292,-0.457982,-0.162605,0.148891,0.710791,1.961230,0.0


,population,stage,s_sigma_label,s_sigma_vol_points,n,mean_c_opt_L,median_c_opt_L,mean_c_opt_S,median_c_opt_S,mean_c_initial_hedge_L,median_c_initial_hedge_L,mean_c_initial_hedge_S,median_c_initial_hedge_S,mean_c_spot_total_L,median_c_spot_total_L,mean_c_spot_total_S,median_c_spot_total_S,median_spot_over_option_L,median_spot_over_option_S,ex_post_friction_materiality_long,ex_post_friction_materiality_short,diagnostic_label
0,ALL_ADMISSIBLE,C,BASE,0.2,961,0.009863,0.009289,0.009863,0.009289,0.000027,0.000023,0.000027,0.000023,0.036789,0.025990,0.036789,0.025990,2.819708,2.819707,0.340271,0.628512,EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY
1,ALL_ADMISSIBLE,C,HIGH,0.4,961,0.019725,0.018579,0.019725,0.018579,0.000027,0.000023,0.000027,0.000023,0.036789,0.025990,0.036789,0.025990,1.409854,1.409854,0.320499,0.611863,EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY
2,ALL_ADMISSIBLE,C,LOW,0.1,961,0.004931,0.004645,0.004931,0.004645,0.000027,0.000023,0.000027,0.000023,0.036789,0.025990,0.036789,0.025990,5.639415,5.639415,0.349636,0.639958,EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY
3,ALL_ADMISSIBLE,VALIDATION,BASE,0.2,961,0.015377,0.015244,0.015377,0.015244,0.000027,0.000021,0.000027,0.000021,0.049758,0.034924,0.049758,0.034924,2.260159,2.260159,0.443288,0.497399,EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY
4,ALL_ADMISSIBLE,VALIDATION,HIGH,0.4,961,0.030755,0.030488,0.030755,0.030488,0.000027,0.000021,0.000027,0.000021,0.049758,0.034924,0.049758,0.034924,1.130080,1.130080,0.415193,0.466181,EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY
5,ALL_ADMISSIBLE,VALIDATION,LOW,0.1,961,0.007689,0.007622,0.007689,0.007622,0.000027,0.000021,0.000027,0.000021,0.049758,0.034924,0.049758,0.034924,4.520318,4.520318,0.460978,0.508845,EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY
6,HIDDEN_EPISODE_ENTRY_ADMISSIBLE,C,BASE,0.2,69,0.013040,0.012682,0.013040,0.012682,0.000025,0.000019,0.000025,0.000019,0.043697,0.026810,0.043697,0.026810,2.195558,2.195557,0.405797,0.579710,EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY
7,HIDDEN_EPISODE_ENTRY_ADMISSIBLE,C,HIGH,0.4,69,0.026079,0.025365,0.026079,0.025365,0.000025,0.000019,0.000025,0.000019,0.043697,0.026810,0.043697,0.026810,1.097779,1.097779,0.362319,0.550725,EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY
8,HIDDEN_EPISODE_ENTRY_ADMISSIBLE,C,LOW,0.1,69,0.006520,0.006341,0.006520,0.006341,0.000025,0.000019,0.000025,0.000019,0.043697,0.026810,0.043697,0.026810,4.391115,4.391115,0.420290,0.579710,EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY
9,HIDDEN_EPISODE_ENTRY_ADMISSIBLE,VALIDATION,BASE,0.2,34,0.015611,0.014565,0.015611,0.014565,0.000023,0.000021,0.000023,0.000021,0.041254,0.033983,0.041254,0.033983,2.120274,2.120274,0.500000,0.411765,EX_POST_FRICTION_MATERIALITY_DIAGNOSTIC_ONLY


These ex-post friction materiality diagnostics are not trade counts. No TEST economic result is displayed.


C:\Users\Rajiv Nawal\AppData\Local\Temp\ipykernel_37424\720289906.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nonfriday_no_entry_audit = panel.assign(no_entry=panel.all_rejection_reasons.str.contains('NO_ENTRY_QUOTE', na=False)).groupby('stage').apply(lambda x: int((~x.is_friday_state_boundary & x.no_entry).sum())).reset_index(name='nonfriday_no_entry_quote_count')


...

,stage,attempted,admissible,rejected
0,C,1209,961,248
1,TEST,1209,962,247
2,VALIDATION,1209,961,248


,stage,n22_episode_entries,mechanically_admissible,mechanically_rejected
0,C,87,69,18
1,TEST,85,74,11
2,VALIDATION,46,34,12


,check,pass,phaseA_ready
0,source_rows_C_validation_test_equal_1209,True,True
1,n22_episode_entries_pre_filter_match,True,True
2,timezone_aware_ny_boundaries,True,True
3,iv_source_key_unique,True,True
4,iv_merge_cardinality_one_to_one,True,True
5,accepted_iv_matches_state_model_day,True,True
6,accepted_has_raw_iv,True,True
7,friday_derived_from_state_boundary,True,True
8,accepted_entry_delay_at_most_60_minutes,True,True
9,accepted_expiry_error_at_most_60_minutes,True,True


,check,pass
0,phase_a_ready_before_pricing,True
1,positive_entry_mid_premium,True
2,positive_scenario_bid_volatility,True
3,nonnegative_terminal_payoff,True
4,gross_pnl_identity,True
5,normalisation_identity,True
6,long_short_gross_symmetry,True
7,option_spread_charged_once_at_entry,True
8,spot_cost_ledger_has_initial_rebalance_and_ter...,True
9,entry_hurdle_dependency_audit_pass,True


PHASE A: PASS | PHASE B: PASS | upstream notebooks modified: NO | TEST economic performance summary displayed: NO | N23 ready for Notebook 24: YES


## 23.12 Exports and final closure

Export the panels and independently derived closure audits.